<a href="https://colab.research.google.com/github/CanerSivri/science-and-art/blob/Week-6/week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install transformers torch

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
def generate_response(user_input):
    """
    Generates a response to user input using the loaded model and tokenizer.

    Args:
        user_input: A string containing the user's prompt.

    Returns:
        A string containing the generated response.
    """
    inputs = tokenizer(user_input, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=100, num_beams=5, early_stopping=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [ ]:
from ipywidgets import Text, Button, VBox, Output
from IPython.display import display

# Create widgets
text_input = Text(description="You:")
button = Button(description="Send")
output_area = Output()

In [ ]:
def on_button_clicked(b):
    """Handles the button click event to generate and display chatbot response."""
    with output_area:
        output_area.clear_output()
        user_input = text_input.value
        if user_input:
            print(f"You: {user_input}")
            response = generate_response(user_input)
            print(f"Chatbot: {response}")
        else:
            print("Please enter some text.")

# Link the button click event to the function
button.on_click(on_button_clicked)

In [ ]:
# Arrange widgets in a VBox and display
chat_interface = VBox([text_input, button, output_area])
display(chat_interface)

In [ ]:
%pip install streamlit
!npm install -g localtunnel

In [ ]:
%%writefile app.py
import streamlit as st
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

@st.cache_resource
def load_model():
    """Loads the pre-trained model and tokenizer."""
    model_name = "google/flan-t5-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return tokenizer, model

# Load model and tokenizer outside the main function to avoid reloading on every interaction
tokenizer, model = load_model()

def generate_response(user_input):
    """Generates a response to user input using the loaded model and tokenizer."""
    inputs = tokenizer(user_input, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=100, num_beams=5, early_stopping=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

st.title("Simple Chatbot")

# Use a session state variable to store the conversation history
if 'messages' not in st.session_state:
    st.session_state.messages = []

# Display conversation history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Get user input
user_input = st.chat_input("You:")

if user_input:
    # Add user message to history
    st.session_state.messages.append({"role": "user", "content": user_input})
    # Display user message
    with st.chat_message("user"):
        st.markdown(user_input)

    # Generate and display chatbot response
    with st.chat_message("assistant"):
        response = generate_response(user_input)
        st.markdown(response)
        # Add chatbot message to history
        st.session_state.messages.append({"role": "assistant", "content": response})

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
# Install ngrok
%pip install pyngrok

In [ ]:
# Run Streamlit with ngrok
from pyngrok import ngrok
import os
import asyncio

# Terminate open tunnels if any
print("Terminating open ngrok tunnels...")
ngrok.kill()

# Set your authtoken (replace with your actual authtoken)
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# It's recommended to store this as a Colab Secret named 'NGROK_AUTH_TOKEN'
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("NGROK_AUTH_TOKEN not found in environment variables. Please set it as a Colab Secret.")


# Define the async function to run Streamlit and ngrok
async def run_streamlit():
    # Start ngrok tunnel
    public_url = ngrok.connect(8501).public_url
    print(f"Streamlit App URL: {public_url}")

    # Run Streamlit app (this will block until Streamlit stops)
    # We'll use asyncio to run it in the background
    process = await asyncio.create_subprocess_shell(
        "streamlit run app.py",
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE
    )

    # Wait for the process to finish and print output (optional)
    stdout, stderr = await process.communicate()
    print(f"Streamlit stdout:\n{stdout.decode()}")
    print(f"Streamlit stderr:\n{stderr.decode()}")

# Run the async function
await run_streamlit()